# DW-1 · Fast-DetectGPT и калибровка EN/RU

Этот ноутбук считает **настоящие скоры** готовой модели на публичных CoAT и HC3.
Он не обучает большую LLM: вычисляет критерий Fast-DetectGPT и калибрует его по языкам.

**Среда выполнения → Сменить среду выполнения → T4 GPU.** Затем «Выполнить все».
Платная подписка не требуется, но доступность бесплатного GPU зависит от квоты Google.
Приватные письма и Google Drive не используются. Итоговый архив содержит скоры и отчёт, без текстов.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path


# Вывод дочерних команд виден в Colab, включая причины ошибок установки.
def run(command):
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    for line in process.stdout:
        print(line.rstrip(), flush=True)
    code = process.wait()
    if code:
        raise subprocess.CalledProcessError(code, command)

ROOT = Path('/content/DW-1')
REPO = 'https://github.com/spotyyyi123-ux/DW-1.git'
BRANCH = 'w1/llm-detectors-calibration'
if not ROOT.exists():
    run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO, str(ROOT)])
os.chdir(ROOT)
run(['git', 'pull', '--ff-only'])
os.environ['PYTHONPATH'] = str(ROOT / 'src')
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'src'))
run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
     'scripts/external/requirements.txt'])
print('Код:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
import torch

assert torch.cuda.is_available(), 'Выберите T4 GPU в меню «Среда выполнения → Сменить среду выполнения».'
print('GPU:', torch.cuda.get_device_name(0))
print('torch:', torch.__version__)

# Проверка аналитической формулы на маленьком известном примере.
import numpy as np
from scipy.special import log_softmax, softmax

from scripts.external.run_fast_detectgpt import discrepancy

x = np.array([[[1., 2., -1.], [0., 1., 3.]]], dtype=np.float32)
y = np.array([[1, 2]])
lp, pr = log_softmax(x, axis=-1), softmax(x, axis=-1)
mean = (pr * lp).sum(-1)
var = (pr * lp**2).sum(-1) - mean**2
expected = (np.take_along_axis(lp, y[..., None], axis=-1).sum() - mean.sum()) / np.sqrt(var.sum())
actual = discrepancy(torch.tensor(x), torch.tensor(y))
assert np.isclose(actual, expected, atol=1e-5), (actual, expected)
print('Самопроверка формулы пройдена.')


## 1. Публичная выборка

Скачаются только CoAT и HC3. По умолчанию: по 50 текстов каждого класса на язык и часть,
около 600 текстов. Вопрос HC3 остаётся в одной части. Для CoAT неизвестны исходные группы
всех вариантов — эта остаточная неопределённость отмечается в отчёте.


In [ ]:
for dataset in ['coat', 'hc3-en']:
    run([sys.executable, 'scripts/fetch_datasets.py', dataset])
run([sys.executable, 'scripts/prepare_llm_benchmark.py', '--per-class', '50'])
import json

import pandas as pd

manifest = json.loads(Path('data/sprint2/llm_benchmark_manifest.json').read_text())
display(pd.DataFrame(manifest['counts']))
print('Текстов:', manifest['n_examples'])


## 2. Fast-DetectGPT на GPU

Используем GPT-Neo 2.7B как одну surrogate-модель для sampling и scoring,
как в базовой демонстрации авторов Fast-DetectGPT. Это не Binoculars и не TF-IDF-бейзлайн.
Модель преимущественно английская: качество и калибровка русского измеряются отдельно.
Оцениваются первые 512 токенов; короткие примеры пропускаются с записью причины.


In [ ]:
run([
    sys.executable, 'scripts/external/run_fast_detectgpt.py',
    '--revision', 'e24fa291132763e59f4a5422741b424fb5d59056',
    '--max-tokens', '512',
])
print('Скоры сохранены без исходных текстов.')


## 3. Калибровка и ECE

Критерий детектора — не вероятность, поэтому напрямую считать для него ECE нельзя.
«До»: общая шкала Platt, обученная только на train. «После»: отдельные isotonic-модели EN/RU,
обученные только на validation. Оценка обоих вариантов — на test.
Изотоническая калибровка не обязана улучшить ECE на маленьком тесте: результат не подгоняем.


In [ ]:
run([sys.executable, '-m', 'dw1.eval.llm_calibration'])
metrics = json.loads(Path('data/runs/sprint2/llm_calibration_metrics.json').read_text())
display(pd.DataFrame([
    dict(язык=lang, тестовых_текстов=metrics[lang]['n_test'],
         ECE_до=metrics[lang]['before']['ece'], ECE_после=metrics[lang]['after']['ece'])
    for lang in ['en', 'ru']
]))


## 4. Скачать результат

Сохрани архив. Его содержимое нужно перенести в локальную папку `data/runs/sprint2/`.
Модель GPT-Neo и исходные тексты в архив не входят. ZIP содержит скоры, метрики и калибраторы.


In [ ]:
from zipfile import ZIP_DEFLATED, ZipFile

from google.colab import files

archive = Path('/content/dw1-sprint2-results.zip')
with ZipFile(archive, 'w', ZIP_DEFLATED) as z:
    for item in sorted(Path('data/runs/sprint2').glob('llm_*')):
        if item.is_file() and '.partial.' not in item.name:
            z.write(item, arcname=item.name)
files.download(str(archive))
